# Gym Weight Loss Prediction - Machine Learning Analysis

## Project Overview
This notebook performs complete ML analysis on gym members data to predict weight loss.

### Steps:
1. Data Loading & Exploration
2. Data Cleaning
3. Exploratory Data Analysis (EDA)
4. Feature Selection
5. Model Training
6. Model Evaluation
7. Model Export

## 1. Import Libraries

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully!')

## 2. Data Loading & Initial Exploration

In [ ]:
# Load the dataset
df = pd.read_csv('../data/gym_members_dataset.csv')

# Display basic information
print('Dataset Shape:', df.shape)
print('\nFirst 10 rows:')
df.head(10)

In [ ]:
# Dataset info and statistics
print('Dataset Info:')
print('=' * 50)
df.info()
print('\nStatistical Summary:')
df.describe().round(2)

## 3. Data Cleaning

In [ ]:
# Check for missing values
print('Missing Values:')
print('=' * 50)
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

# Check for duplicates
duplicates = df.duplicated().sum()
print(f'\nNumber of duplicate rows: {duplicates}')
if duplicates > 0:
    df = df.drop_duplicates()
    print(f'Duplicates removed. New shape: {df.shape}')
else:
    print('No duplicates found.')

# Check data types and negative values
print('\nData Types:')
print(df.dtypes)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['weight_loss_kg'], bins=20, color='#667eea',
             edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Weight Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Weight Loss (kg)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['weight_loss_kg'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["weight_loss_kg"].mean():.2f}')
axes[0].legend()

axes[1].boxplot(df['weight_loss_kg'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#667eea', alpha=0.7))
axes[1].set_title('Weight Loss Box Plot', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Weight Loss (kg)')

plt.tight_layout()
os.makedirs('../static/images', exist_ok=True)
plt.savefig('../static/images/weight_loss_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlBu_r', center=0, square=True, linewidths=1)
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../static/images/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelation with Weight Loss:')
print('=' * 40)
target_corr = correlation_matrix['weight_loss_kg'].drop('weight_loss_kg').sort_values(ascending=False)
for feature, corr in target_corr.items():
    print(f'  {feature:30s}: {corr:.4f}')

In [ ]:
# Weight loss by gender
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gender_loss = df.groupby('gender')['weight_loss_kg'].mean()
bars = axes[0].bar(['Female', 'Male'], gender_loss.values,
                    color=['#eb3349', '#667eea'], edgecolor='white', width=0.5)
axes[0].set_title('Average Weight Loss by Gender', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Average Weight Loss (kg)')
for bar, val in zip(bars, gender_loss.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
                f'{val:.2f} kg', ha='center', fontweight='bold')

gender_counts = df['gender'].value_counts()
axes[1].pie(gender_counts.values, labels=['Male', 'Female'],
            colors=['#667eea', '#eb3349'], autopct='%1.1f%%',
            startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Gender Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../static/images/gender_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Workout hours vs Weight loss scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scatter = axes[0].scatter(df['workout_hours_per_week'], df['weight_loss_kg'],
                          c=df['gender'], cmap='coolwarm', alpha=0.7, edgecolors='white')
axes[0].set_title('Workout Hours vs Weight Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Workout Hours per Week')
axes[0].set_ylabel('Weight Loss (kg)')
z = np.polyfit(df['workout_hours_per_week'], df['weight_loss_kg'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['workout_hours_per_week'].min(),
                     df['workout_hours_per_week'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', alpha=0.8, linewidth=2, label='Trend')
axes[0].legend()

scatter2 = axes[1].scatter(df['daily_calorie_intake'], df['weight_loss_kg'],
                           c=df['age'], cmap='viridis', alpha=0.7, edgecolors='white')
axes[1].set_title('Calorie Intake vs Weight Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Daily Calorie Intake')
axes[1].set_ylabel('Weight Loss (kg)')
plt.colorbar(scatter2, ax=axes[1], label='Age')

plt.tight_layout()
plt.savefig('../static/images/scatter_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Selection

In [ ]:
# Feature importance using Random Forest
feature_names = ['age', 'gender', 'height_cm', 'weight_kg',
                 'workout_hours_per_week', 'daily_calorie_intake', 'sleep_hours']
target = 'weight_loss_kg'

X = df[feature_names]
y = df[target]

rf_temp = RandomForestRegressor(n_estimators=100, random_state=42)
rf_temp.fit(X, y)

importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_temp.feature_importances_
}).sort_values('Importance', ascending=False)

print('Feature Importance (Random Forest):')
print(importance.to_string(index=False))

plt.figure(figsize=(10, 6))
bars = plt.barh(importance['Feature'], importance['Importance'],
                color='#667eea', edgecolor='white', height=0.6)
plt.xlabel('Importance Score')
plt.title('Feature Importance for Weight Loss Prediction',
           fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
for bar, val in zip(bars, importance['Importance']):
    plt.text(val + 0.005, bar.get_y() + bar.get_height()/2.,
             f'{val:.3f}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../static/images/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Model Training

In [ ]:
# Prepare data for training
X = df[feature_names].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set: {X_test.shape[0]} samples')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('\nFeatures scaled using StandardScaler.')

In [ ]:
# Train multiple models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=100, random_state=42, max_depth=10),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100, random_state=42, max_depth=5),
    'SVR': SVR(kernel='rbf', C=100, gamma='scale'),
}

results = {}
for name, model in models.items():
    print(f'\n--- Training {name} ---')
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    
    results[name] = {
        'model': model, 'y_pred': y_pred,
        'mae': round(mae, 4), 'mse': round(mse, 4),
        'rmse': round(rmse, 4), 'r2': round(r2, 4),
        'cv_mean': round(cv_scores.mean(), 4),
    }
    print(f'  MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}')
    print(f'  CV R2 Mean: {cv_scores.mean():.4f}')

## 7. Model Evaluation & Comparison

In [ ]:
# Model comparison
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'MAE': [results[m]['mae'] for m in results],
    'RMSE': [results[m]['rmse'] for m in results],
    'R2 Score': [results[m]['r2'] for m in results],
})
print('Model Comparison:')
print(comparison.to_string(index=False))

best_model_name = comparison.loc[comparison['R2 Score'].idxmax(), 'Model']
print(f'\nBest Model: {best_model_name}')

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
model_names = list(results.keys())
r2_scores = [results[m]['r2'] for m in model_names]
rmse_scores = [results[m]['rmse'] for m in model_names]
colors = ['#667eea', '#11998e', '#eb3349', '#764ba2']

axes[0].bar(model_names, r2_scores, color=colors, edgecolor='white', width=0.6)
axes[0].set_title('R2 Score Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('R2 Score')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(model_names, rmse_scores, color=colors, edgecolor='white', width=0.6)
axes[1].set_title('RMSE Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('RMSE')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../static/images/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Actual vs Predicted for best model
best_result = results[best_model_name]
y_pred_best = best_result['y_pred']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_best, alpha=0.7, color='#667eea', edgecolors='white')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2)
axes[0].set_title(f'Actual vs Predicted ({best_model_name})',
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel('Actual Weight Loss (kg)')
axes[0].set_ylabel('Predicted Weight Loss (kg)')

residuals = y_test - y_pred_best
axes[1].hist(residuals, bins=15, color='#764ba2', edgecolor='white', alpha=0.8)
axes[1].set_title('Residual Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].axvline(0, color='red', linestyle='--')

plt.tight_layout()
plt.savefig('../static/images/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Model

In [ ]:
# Save the best model and scaler
os.makedirs('../ml_model', exist_ok=True)

best_model = results[best_model_name]['model']
joblib.dump(best_model, '../ml_model/weight_loss_model.pkl')
joblib.dump(scaler, '../ml_model/scaler.pkl')

metrics = {
    'best_model': best_model_name,
    'r2_score': best_result['r2'],
    'mae': best_result['mae'],
    'mse': best_result['mse'],
    'rmse': best_result['rmse'],
}
with open('../ml_model/model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print('Model saved successfully!')
print(f'Best Model: {best_model_name}')
print(f'R2 Score: {best_result["r2"]}')
print(f'MAE: {best_result["mae"]}')
print(f'RMSE: {best_result["rmse"]}')